# 拆分 StockProfitability_all.xlsx → 每支股票一個檔案

**輸入**：`Result/StockProfitability_all.xlsx`  
**輸出**：`Result/profitability/{stock_number}.xlsx`  

每支股票會產生一個獨立的 xlsx，包含該股所有季度財報資料。

In [1]:
import pandas as pd
import os
from pathlib import Path

# ── 設定路徑 ──────────────────────────────────────────────
INPUT_FILE  = Path('Result/StockProfitability_all.xlsx')
OUTPUT_DIR  = Path('Result/profitability')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 讀入總表 ──────────────────────────────────────────────
print('讀取中...')
df = pd.read_excel(INPUT_FILE, dtype={'stock_number': str})

# stock_number 補齊 4 位（例如 0050）
df['stock_number'] = df['stock_number'].str.zfill(4)

stocks = df['stock_number'].unique()
print(f'共 {len(stocks)} 支股票，開始拆分...')

讀取中...
共 1965 支股票，開始拆分...


In [2]:
# ── 拆分並寫出 ────────────────────────────────────────────
from tqdm.auto import tqdm  # pip install tqdm（沒有也可以拿掉）

skipped = []

for stock in tqdm(stocks):
    sub = df[df['stock_number'] == stock].copy()

    # 依季別排序（season_key 已是數字，直接排）
    if 'season_key' in sub.columns:
        sub = sub.sort_values('season_key').reset_index(drop=True)

    out_path = OUTPUT_DIR / f'{stock}.xlsx'
    try:
        sub.to_excel(out_path, index=False)
    except Exception as e:
        skipped.append((stock, str(e)))

print(f'\n完成！寫出 {len(stocks) - len(skipped)} 個檔案')
if skipped:
    print(f'失敗 {len(skipped)} 個：', skipped)

  0%|          | 0/1965 [00:00<?, ?it/s]


完成！寫出 1965 個檔案


In [3]:
# ── 驗證：抽查一支 ────────────────────────────────────────
sample = stocks[0]
check = pd.read_excel(OUTPUT_DIR / f'{sample}.xlsx')
print(f'股票 {sample}：{len(check)} 季')
check[['stock_number', '季別', 'EPS(元)', '毛利率', '去年同季比較']].tail(8)

股票 9962：73 季


,stock_number,季別,EPS(元),毛利率,去年同季比較
65,9962,113.2Q,0.31,8.66,EPS衰退 / 毛利率成長
66,9962,113.3Q,0.18,8.06,EPS衰退 / 毛利率成長
67,9962,113.4Q,0.12,5.06,EPS衰退 / 毛利率衰退
68,9962,114.1Q,0.12,6.05,EPS成長 / 毛利率成長
69,9962,114.2Q,-0.28,0.77,EPS衰退 / 毛利率衰退
70,9962,114.3Q,-0.07,3.06,EPS衰退 / 毛利率衰退
71,9962,114.4Q,0.08,6.35,EPS衰退 / 毛利率成長
72,9962,115.1Q,0.05,7.63,EPS衰退 / 毛利率成長


In [4]:
# ── 統計摘要 ──────────────────────────────────────────────
sizes = {}
for f in OUTPUT_DIR.glob('*.xlsx'):
    sizes[f.stem] = pd.read_excel(f, usecols=[0]).shape[0]

s = pd.Series(sizes)
print(f'總檔案數  : {len(s)}')
print(f'平均季數  : {s.mean():.1f}')
print(f'最多季    : {s.max()}  ({s.idxmax()})')
print(f'最少季    : {s.min()}  ({s.idxmin()})')

總檔案數  : 1965
平均季數  : 60.9
最多季    : 97  (2883)
最少季    : 5  (3717)
